# Metodologia Design Science Research (DSR)

**Etapa de Pesquisa (Peffers et al., 2007):**
### 3. Design e Desenvolvimento (Design and Development)

**Objetivo Acadêmico:** Este notebook foca na criação da base de dados inicial e anonimização de identificadores. É a fase de construção dos alicerces do artefato, garantindo a integridade dos dados e a conformidade ética (anonimização) necessária para um sistema de gestão pública sob a ótica da Indústria 5.0.


# 01a - Ingestão e Anonimização de Reservas

Este notebook processa o arquivo bruto de reservas, anonimizando a identidade dos alunos via SHA-256.

In [1]:
import pandas as pd
import hashlib
import os
import glob
import unicodedata

# 1. Configurações de Segurança e Pastas
CHAVE_SEGURANCA = "predicao_refeicoes_v1"
DIRETORIO_BRUTO_PATTERN = "../data/reserva*/" 

def normalizar_texto(s):
    if not isinstance(s, str): return str(s)
    # Remove acentos, espaços extras e converte para minúsculo
    n = unicodedata.normalize('NFD', s).encode('ascii', 'ignore').decode('utf-8')
    return n.lower().strip()

def anonimizar_id(matricula):
    if pd.isna(matricula): return None
    return hashlib.sha256((str(matricula) + CHAVE_SEGURANCA).encode()).hexdigest()[:12]

# 2. Motor de Ingestão Inteligente (Fuzzy Mapping)
def carregar_e_mapear_colunas(caminho, mapa_obrigatorio):
    try:
        df_bruto = pd.read_excel(caminho)
    except Exception as e:
        print(f"❌ Erro ao abrir {os.path.basename(caminho)}: {e}")
        return None

    # Normalizar as colunas que estão no Excel agora
    colunas_excel = df_bruto.columns.tolist()
    norm_cols_excel = {normalizar_texto(c): c for c in colunas_excel}
    
    mapeamento_final = {}
    faltantes = []

    for nome_interno, sinonimos in mapa_obrigatorio.items():
        achou = False
        for sin in sinonimos:
            # Busca o sinônimo normalizado no dicionário de colunas normalizadas do Excel
            if normalizar_texto(sin) in norm_cols_excel:
                mapeamento_final[nome_interno] = norm_cols_excel[normalizar_texto(sin)]
                achou = True
                break
        if not achou: faltantes.append(nome_interno)
    
    if faltantes:
        print(f"⚠️ Aviso: Arquivo {os.path.basename(caminho)} ignorado. Faltam: {faltantes}")
        return None
    
    # Seleção e Renomeação utilizando os nomes ORIGINAIS encontrados no Excel
    df = df_bruto[list(mapeamento_final.values())].copy()
    df.columns = [k for k in mapeamento_final.keys()]
    return df

# 3. Execução em Lote
MAPA = {
    'MATRICULA': ['matricula iq', 'matricula i q', 'matricula', 'iq', 'codigo', 'id'],
    'DATA': ['data', 'dt', 'dia'],
    'NOME': ['nome', 'aluno', 'estudante', 'nome do aluno'],
    'TURMA': ['turma', 'classe', 'ano', 'serie'],
    'REFEICAO': ['refeicao', 'prato', 'escolha', 'opcao', 'tipo de refeicao']
}

arquivos = glob.glob(os.path.join(DIRETORIO_BRUTO_PATTERN, "*.xlsx"))
if not arquivos:
    # Fallback caso a pasta tenha o nome exato sem o plural/pattern
    arquivos = glob.glob(os.path.join("../data/reservas/", "*.xlsx"))

print(f"🔍 Encontrados {len(arquivos)} arquivos para processar.")

lista_dfs = []
for f in arquivos:
    print(f"📖 Lendo: {os.path.basename(f)}...")
    df_temp = carregar_e_mapear_colunas(f, MAPA)
    if df_temp is not None:
        exclusoes = ["NAO IRA ALMOCAR", "SEM RESERVA", "NAN", "", "NONE", "0"]
        df_temp["norm_ref"] = df_temp["REFEICAO"].apply(normalizar_texto).str.upper()
        df_efetivo = df_temp[~df_temp["norm_ref"].isin(exclusoes)].copy()
        lista_dfs.append(df_efetivo)

if lista_dfs:
    df_unificado = pd.concat(lista_dfs).drop_duplicates()
    
    # 4. Tratamento de Datas e Tradução
    df_unificado["DATA"] = pd.to_datetime(df_unificado["DATA"], dayfirst=True, errors='coerce')
    df_unificado = df_unificado.dropna(subset=["DATA"])
    
    dias_ptbr = { 'Monday': 'Segunda-feira', 'Tuesday': 'Terça-feira', 'Wednesday': 'Quarta-feira', 'Thursday': 'Quinta-feira', 'Friday': 'Sexta-feira', 'Saturday': 'Sábado', 'Sunday': 'Domingo' }
    df_unificado["dia_semana"] = df_unificado["DATA"].dt.day_name().map(dias_ptbr)
    
    # 5. Anonimização
    df_unificado["id_aluno_anonimo"] = df_unificado["MATRICULA"].apply(anonimizar_id)
    
    # 6. Exportação Final
    df_export = df_unificado[["DATA", "id_aluno_anonimo", "TURMA", "REFEICAO", "dia_semana"]].copy()
    df_export.columns = ["data", "id_aluno_anonimo", "turma", "categoria_reserva", "dia_semana"]
    if "reservas" == "consumo": df_export["servida"] = 1
    
    caminho_saida = "../data/reservas_anonimo.csv" if "reservas" == "consumo" else "../data/reservas_anonimas.csv"
    df_export.to_csv(caminho_saida, index=False)
    print(f"🚀 SUCESSO! Base unificada salva em: {caminho_saida}")
    display(df_export.head())
else:
    print("⚠️ Nenhum arquivo válido processado.")


🔍 Encontrados 1 arquivos para processar.
📖 Lendo: reservas_2025.xlsx...


🚀 SUCESSO! Base unificada salva em: ../data/reservas_anonimas.csv


,data,id_aluno_anonimo,turma,categoria_reserva,dia_semana
0,2025-02-24,772130313177,1º A - MEC,PRATO TRADICIONAL (A OPÇÃO PROTÉICA É CARNE),Segunda-feira
2,2025-02-24,a9d0dd49108a,1º A - MEC,PRATO TRADICIONAL (A OPÇÃO PROTÉICA É CARNE),Segunda-feira
4,2025-02-24,47ea4259aebe,1º A - MEC,PRATO TRADICIONAL (A OPÇÃO PROTÉICA É CARNE),Segunda-feira
5,2025-02-24,040eef61e258,1º A - MEC,PRATO TRADICIONAL (A OPÇÃO PROTÉICA É CARNE),Segunda-feira
6,2025-02-24,33acb4bc7754,1º A - MEC,PRATO TRADICIONAL (A OPÇÃO PROTÉICA É CARNE),Segunda-feira


# Sumário da Ingestão de Reservas
Resumo quantitativo e estatístico dos dados processados.

In [2]:
# Sumário Simplificado (Apenas Texto e Tabela)
import pandas as pd
df_temp = df_export.copy()
total_registros = len(df_temp)
total_dias = df_temp['data'].nunique()
primeira_data = df_temp['data'].min().strftime('%d/%m/%Y')
ultima_data = df_temp['data'].max().strftime('%d/%m/%Y')

print(f"--- RELATÓRIO DE INGESTÃO (RESERVAS) ---")
print(f"Total de Registros: {total_registros}")
print(f"Total de Dias: {total_dias}")
print(f"Período: {primeira_data} até {ultima_data}")
print(f"Volume Médio Diário (Geral): {total_registros/total_dias:.1f} registros/dia")

# 1. Volume Médio por Dia da Semana
ordem_dias = ['Segunda-feira', 'Terça-feira', 'Quarta-feira', 'Quinta-feira', 'Sexta-feira']
avg_weekday = df_temp.groupby(['data', 'dia_semana']).size().reset_index(name='qtd')
avg_weekday_mean = avg_weekday.groupby('dia_semana')['qtd'].mean().reindex(ordem_dias)

print("\nVolume Médio por Dia da Semana:")
for dia, media in avg_weekday_mean.items():
    if not pd.isna(media):
        print(f" - {dia}: {media:.1f}")

# 2. Visão Detalhada: Todas as Datas e Quantidades
print("\nDetalhamento por Data (Cronológico):")
detalhe_datas = df_temp.groupby(['data', 'dia_semana']).size().reset_index(name='quantidade')
detalhe_datas.columns = ['Data', 'Dia da Semana', 'Quantidade']
display(detalhe_datas.style.background_gradient(cmap='Greens', subset=['Quantidade']))

# 3. Visão Mensal Rápida
df_temp['ano_mes'] = df_temp['data'].dt.strftime('%Y-%m')
res_por_mes = df_temp.groupby('ano_mes').size().reset_index(name='qtd')
print("\nVolume Consolidado por Mês:")
print(res_por_mes.to_string(index=False))

--- RELATÓRIO DE INGESTÃO (RESERVAS) ---
Total de Registros: 19826
Total de Dias: 177
Período: 24/02/2025 até 12/12/2025
Volume Médio Diário (Geral): 112.0 registros/dia

Volume Médio por Dia da Semana:
 - Segunda-feira: 130.0
 - Terça-feira: 127.0
 - Quarta-feira: 115.4
 - Quinta-feira: 106.9
 - Sexta-feira: 82.1

Detalhamento por Data (Cronológico):


,Data,Dia da Semana,Quantidade
0,2025-02-24 00:00:00,Segunda-feira,167
1,2025-02-25 00:00:00,Terça-feira,76
2,2025-02-26 00:00:00,Quarta-feira,151
3,2025-02-27 00:00:00,Quinta-feira,114
4,2025-02-28 00:00:00,Sexta-feira,118
5,2025-03-05 00:00:00,Quarta-feira,1
6,2025-03-06 00:00:00,Quinta-feira,127
7,2025-03-07 00:00:00,Sexta-feira,67
8,2025-03-10 00:00:00,Segunda-feira,189
9,2025-03-11 00:00:00,Terça-feira,154



Volume Consolidado por Mês:
ano_mes  qtd
2025-02  626
2025-03 2329
2025-04 2643
2025-05 2676
2025-06 2443
2025-07  751
2025-08 1820
2025-09 2431
2025-10 1385
2025-11 1952
2025-12  770
